# Form 13F — `INFOTABLE.tsv` smoke test

Tab-separated holdings from the SEC 13F bulk file. The full `INFOTABLE.tsv` in this download is **~3.5M rows**; set `NROWS` to `None` only when you want the full table in memory (large RAM use).

In [1]:
import pandas as pd

INFOTABLE_PATH = r"C:\Users\mandviv\Downloads\01dec2025-28feb2026_form13f\INFOTABLE.tsv"

NROWS = 10_000  # use None to load the entire file

df = pd.read_csv(INFOTABLE_PATH, sep="\t", nrows=NROWS, low_memory=False)
df

,ACCESSION_NUMBER,INFOTABLE_SK,NAMEOFISSUER,TITLEOFCLASS,CUSIP,FIGI,VALUE,SSHPRNAMT,SSHPRNAMTTYPE,PUTCALL,INVESTMENTDISCRETION,OTHERMANAGER,VOTING_AUTH_SOLE,VOTING_AUTH_SHARED,VOTING_AUTH_NONE
0,0002085853-26-000289,125717297,APPLE INC,COM,037833100,NaN,6362340,23403,SH,NaN,SOLE,NaN,23403,0,0
1,0002085853-26-000289,125717298,BLACKROCK ETF TRUST,ISHARES US LARG,09290C863,NaN,7309780,230738,SH,NaN,SOLE,NaN,230738,0,0
2,0002085853-26-000289,125717299,ETF SER SOLUTIONS,DEFIANCE QUANTUM,26922A420,NaN,2032548,18535,SH,NaN,SOLE,NaN,18535,0,0
3,0002085853-26-000289,125717300,EXXON MOBIL CORP,COM,30231G102,NaN,880287,7315,SH,NaN,SOLE,NaN,7315,0,0
4,0002085853-26-000289,125717301,FIRST TR EXCH TRADED FD III,FT VEST SMID,33738D820,NaN,12765181,594836,SH,NaN,SOLE,NaN,594836,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,0001752724-26-000015,125718970,VERIZON COMMUNICATIONS INC,COM,92343V104,NaN,2330448,57217,SH,NaN,SOLE,NaN,57217,0,0
9996,0001752724-26-000015,125718971,WHEATON PRECIOUS METALS CORP,COM,962879102,NaN,6434875,39879,SH,NaN,SOLE,NaN,39879,0,0
9997,0001752724-26-000015,125718972,KEYCORP,COM,493267108,NaN,598519,28998,SH,NaN,SOLE,NaN,28998,0,0
9998,0001752724-26-000015,125718973,COSTAR GROUP INC,COM,22160N109,NaN,665071,9891,SH,NaN,SOLE,NaN,9891,0,0


In [2]:
print("shape:", df.shape)
df.dtypes


shape: (10000, 15)


ACCESSION_NUMBER          str
INFOTABLE_SK            int64
NAMEOFISSUER              str
TITLEOFCLASS              str
CUSIP                     str
FIGI                      str
VALUE                   int64
SSHPRNAMT               int64
SSHPRNAMTTYPE             str
PUTCALL                   str
INVESTMENTDISCRETION      str
OTHERMANAGER              str
VOTING_AUTH_SOLE        int64
VOTING_AUTH_SHARED      int64
VOTING_AUTH_NONE        int64
dtype: object

In [3]:
df.head(10)

,ACCESSION_NUMBER,INFOTABLE_SK,NAMEOFISSUER,TITLEOFCLASS,CUSIP,FIGI,VALUE,SSHPRNAMT,SSHPRNAMTTYPE,PUTCALL,INVESTMENTDISCRETION,OTHERMANAGER,VOTING_AUTH_SOLE,VOTING_AUTH_SHARED,VOTING_AUTH_NONE
0,0002085853-26-000289,125717297,APPLE INC,COM,037833100,NaN,6362340,23403,SH,NaN,SOLE,NaN,23403,0,0
1,0002085853-26-000289,125717298,BLACKROCK ETF TRUST,ISHARES US LARG,09290C863,NaN,7309780,230738,SH,NaN,SOLE,NaN,230738,0,0
2,0002085853-26-000289,125717299,ETF SER SOLUTIONS,DEFIANCE QUANTUM,26922A420,NaN,2032548,18535,SH,NaN,SOLE,NaN,18535,0,0
3,0002085853-26-000289,125717300,EXXON MOBIL CORP,COM,30231G102,NaN,880287,7315,SH,NaN,SOLE,NaN,7315,0,0
4,0002085853-26-000289,125717301,FIRST TR EXCH TRADED FD III,FT VEST SMID,33738D820,NaN,12765181,594836,SH,NaN,SOLE,NaN,594836,0,0
5,0002085853-26-000289,125717302,FIRST TR EXCHANGE TRADED FD,NASDQ CLN EDGE,33737A108,NaN,2603329,17013,SH,NaN,SOLE,NaN,17013,0,0
6,0002085853-26-000289,125717303,FIRST TR EXCHANGE TRADED FD,RBA INDL ETF,33738R704,NaN,6482041,65928,SH,NaN,SOLE,NaN,65928,0,0
7,0002085853-26-000289,125717304,FIRST TR EXCHANGE-TRADED FD,FT VEST TEC,33738D812,NaN,15139327,545758,SH,NaN,SOLE,NaN,545758,0,0
8,0002085853-26-000289,125717305,INNOVATOR ETFS TRUST,US EQTY PWR BUF,45782C813,NaN,285684,6153,SH,NaN,SOLE,NaN,6153,0,0
9,0002085853-26-000289,125717306,ISHARES BITCOIN TRUST ETF,SHS BEN INT,46438F101,NaN,433643,8734,SH,NaN,SOLE,NaN,8734,0,0


## MSCI — CUSIP `55354G100`

MSCI Inc. common. The full TSV is read in **chunks** so you do not load ~3.5M rows at once. Change `MSCI_CUSIP` if you need a different security.

In [4]:
MSCI_CUSIP = "55354G100"
CHUNKSIZE = 300_000

parts: list[pd.DataFrame] = []
for chunk in pd.read_csv(
    INFOTABLE_PATH,
    sep="\t",
    chunksize=CHUNKSIZE,
    low_memory=False,
    dtype={"CUSIP": "string", "ACCESSION_NUMBER": "string"},
):
    m = chunk[chunk["CUSIP"] == MSCI_CUSIP]
    if not m.empty:
        parts.append(m)

df_msci = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
print("rows:", len(df_msci))
if not df_msci.empty:
    print(df_msci["NAMEOFISSUER"].value_counts().head(10))
df_msci

rows: 1802
NAMEOFISSUER
MSCI INC         1540
MSCI INC COM      121
MSCI Inc           60
MSCI               10
MSCI Inc.           5
MSCI INC-A          5
Msci Inc            4
MSCI INC CL A       4
Msci Inc - US       3
MSCI  INC           3
Name: count, dtype: int64


,ACCESSION_NUMBER,INFOTABLE_SK,NAMEOFISSUER,TITLEOFCLASS,CUSIP,FIGI,VALUE,SSHPRNAMT,SSHPRNAMTTYPE,PUTCALL,INVESTMENTDISCRETION,OTHERMANAGER,VOTING_AUTH_SOLE,VOTING_AUTH_SHARED,VOTING_AUTH_NONE
0,0002043986-26-000004,125718438,MSCI INC,COM,55354G100,NaN,209374,369,SH,NaN,SOLE,NaN,0,0,369
1,0002043986-26-000003,125717539,MSCI INC,COM,55354G100,NaN,220891,383,SH,NaN,SOLE,NaN,0,0,383
2,0002043986-26-000002,125716866,MSCI INC,COM,55354G100,NaN,213194,377,SH,NaN,SOLE,NaN,0,0,377
3,0001752724-26-000015,125718961,MSCI INC,COM,55354G100,NaN,1091234,1902,SH,NaN,SOLE,NaN,1902,0,0
4,0000312069-26-000078,125702869,MSCI INC,COM,55354G100,NaN,183661700,318448,SH,NaN,SOLE,1,318448,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1797,0001172661-25-005234,122127342,MSCI INC,COM,55354G100,NaN,913848,1616,SH,NaN,DFND,1,1616,0,0
1798,0001172661-25-005234,122127343,MSCI INC,COM,55354G100,NaN,2774343,4906,SH,NaN,SOLE,NaN,4906,0,0
1799,0001691766-25-000008,122119330,MSCI INC,COM,55354G100,BBG001SV8B05,213218,443,SH,NaN,SOLE,NaN,0,0,443
1800,0001752724-25-215030,122116786,MSCI INC,COM,55354G100,NaN,1170567,2063,SH,NaN,SOLE,NaN,2063,0,0
